In [1]:
!pip install transformers bitsandbytes accelerate

import importlib, site, sys
importlib.reload(site)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu

<module 'site' (frozen)>

In [2]:
import torch
import time
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import csv
import sys
import re
from tqdm.notebook import tqdm
import os
from kaggle_secrets import UserSecretsClient
from peft import PeftModel, PeftConfig

2026-05-16 19:25:59.041867: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778959559.390691      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778959559.499124      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
model_id       = "mistralai/Mistral-7B-Instruct-v0.3"
adapter_path   = "/kaggle/input/datasets/linhtron123/finetune-mistral/final_model_best"
output_pretrained = "/kaggle/working/results_zeroshot_pretrained.csv"
output_finetuned  = "/kaggle/working/results_zeroshot_finetuned.csv"

allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion",
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

In [4]:
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except:
    hf_token = os.environ.get("HF_TOKEN")

test_df = pd.read_csv('/kaggle/input/datasets/linhtron123/legaldata/test_split.csv')
print(f"Test set loaded: {len(test_df)} rows")

HF token loaded.
Test set loaded: 155 rows


In [5]:
def build_zeroshot_prompt(tokenizer, dialogue):
    messages = [
        {
            "role": "user",
            "content": f"""You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Determine if any manipulation is present or not.
3. If manipulation is present, identify the primary manipulator.
4. If manipulation is present, identify the techniques used from the allowed list ONLY.

IMPORTANT: You can ONLY use these exact techniques (no variations or additions):
{', '.join(allowed_tactics)}

Dialogue: {dialogue}

Question: Is there manipulation present in this dialogue? If yes, identify the primary manipulator and techniques.

Response Format:
Manipulation Present: [Yes/No]
Primary Manipulator: [Plaintiff/Defendant/None]
Techniques: [technique1, technique2, ... / None]"""
        }
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [6]:
def detect_manipulation(statement, pipe, tokenizer):
    prompt       = build_zeroshot_prompt(tokenizer, statement)
    terminators  = [pipe.tokenizer.eos_token_id]

    response = pipe(
        prompt,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        eos_token_id=terminators,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    full_text        = response[0]["generated_text"]
    split_marker     = prompt.split("[/INST]")[-1].strip()
    prompt_end_index = full_text.find(split_marker) + len(split_marker)

    if prompt_end_index > len(split_marker):
        return full_text[prompt_end_index:].strip()
    return full_text.split("[/INST]")[-1].strip()

In [7]:
def process_and_save(pipe, df, tokenizer, output_path, label):
    print(f"\n{'='*60}")
    print(f"  Running inference: {label}")
    print(f"{'='*60}")
    print("Starting analysis...")

    manip_preds, manipulators, all_techs = [], [], []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=label):
        try:
            response_text = detect_manipulation(row["Dialogue"], pipe, tokenizer)
            m, p, t       = extract_analysis_result(response_text)
        except Exception as e:
            print(f"\n!!! LỖI xử lý hàng {idx}: {e}")
            m, p, t = 0, "Error", ["Error"]

        manip_preds.append(m)
        manipulators.append(p)
        all_techs.append(t)

        if (idx + 1) % 5 == 0:
            print(f"\nProcessed {idx+1}/{len(df)} dialogues")

        if (idx + 1) % 20 == 0:
            ckpt_df = df.iloc[:idx+1].copy()
            ckpt_df["Manipulative_Pred"]        = manip_preds
            ckpt_df["Primary_Manipulator_Pred"] = manipulators
            ckpt_df["Techniques_Pred"]          = pd.Series(all_techs).apply(
                lambda x: ", ".join(x) if isinstance(x, list) else "None"
            )
            ckpt_path = output_path.replace(".csv", f"_ckpt{idx+1}.csv")
            ckpt_df.to_csv(ckpt_path, index=False)
            print(f"[Checkpoint saved at row {idx+1} → {ckpt_path}]\n")

    result_df = df.copy()
    result_df["Manipulative_Pred"]        = manip_preds
    result_df["Primary_Manipulator_Pred"] = manipulators
    result_df["Techniques_Pred"]          = pd.Series(all_techs).apply(
        lambda x: ", ".join(x) if isinstance(x, list) else "None"
    )
    result_df.to_csv(output_path, index=False)
    print(f"\n[Done] Saved → {output_path}")
    return result_df

In [8]:
def extract_analysis_result(response):
    Manipulative_Pred   = 0
    primary_manipulator = "None"
    techniques          = []

    present_match = re.search(r"Manipulation Present:\s*(Yes|No)", response, re.IGNORECASE)
    if present_match and present_match.group(1).lower() == "yes":
        Manipulative_Pred = 1
    else:
        print(f"Final extracted values: Manipulative=0, Manipulator=None, Techniques=None")
        print("-" * 80)
        return 0, "None", ["None"]

    manipulator_match = re.search(r"Primary Manipulator:\s*([^\n]+)", response, re.IGNORECASE)
    if manipulator_match:
        txt = manipulator_match.group(1).strip().lower()
        if "plaintiff" in txt:
            primary_manipulator = "Plaintiff"
        elif "defendant" in txt:
            primary_manipulator = "Defendant"

    techniques_match = re.search(r"Techniques:\s*([^\n]+)", response, re.IGNORECASE)
    if techniques_match:
        techniques_text = techniques_match.group(1).strip()
        if techniques_text.lower() not in ["none", "n/a", "", "[]"]:
            allowed_lower = [a.lower() for a in allowed_tactics]
            for raw_tech in [t.strip().lower() for t in techniques_text.split(",")]:
                if raw_tech in allowed_lower:
                    techniques.append(allowed_tactics[allowed_lower.index(raw_tech)])

    techniques_list_or_none = techniques if techniques else ["None"]
    print(f"Final extracted values: Manipulative={Manipulative_Pred}, Manipulator={primary_manipulator}, Techniques={techniques_list_or_none}")
    print("-" * 80)
    return Manipulative_Pred, primary_manipulator, techniques_list_or_none

In [9]:
def evaluate(result_df, label):
    print(f"\n{'='*60}")
    print(f"  EVALUATION: {label}")
    print(f"{'='*60}")

    # --- Task 1: Binary ---
    targets_bin = result_df["Manipulative"].astype(int).tolist()
    preds_bin   = result_df["Manipulative_Pred"].astype(int).tolist()

    print("\n--- Task 1: Binary Classification ---")
    print(f"Accuracy:  {accuracy_score(targets_bin, preds_bin):.3f}")
    print(f"Precision: {precision_score(targets_bin, preds_bin, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(targets_bin, preds_bin, zero_division=0):.3f}")
    print(f"F1 Score:  {f1_score(targets_bin, preds_bin, average='binary', zero_division=0):.3f}")
    print(f"Confusion Matrix:\n{confusion_matrix(targets_bin, preds_bin)}")

    # --- Task 2: Multi-class ---
    gt_manip   = result_df["Primary Manipulator"].fillna("None").str.strip().str.capitalize()
    pred_manip = result_df["Primary_Manipulator_Pred"].fillna("None").str.strip().str.capitalize()

    all_cats = sorted(set(gt_manip.tolist() + pred_manip.tolist()))
    enc = LabelEncoder()
    enc.fit(all_cats)

    print("\n--- Task 2: Multi-class Classification ---")
    print(f"Label mapping: {dict(zip(enc.classes_, enc.transform(enc.classes_)))}")
    tgt_enc  = enc.transform(gt_manip)
    pred_enc = enc.transform(pred_manip)

    print(f"Accuracy:            {accuracy_score(tgt_enc, pred_enc):.3f}")
    print(f"Precision (Weighted):{precision_score(tgt_enc, pred_enc, average='weighted', zero_division=0):.3f}")
    print(f"Recall    (Weighted):{recall_score(tgt_enc, pred_enc, average='weighted', zero_division=0):.3f}")
    print(f"F1 Score  (Weighted):{f1_score(tgt_enc, pred_enc, average='weighted', zero_division=0):.3f}")
    print(f"Confusion Matrix:\n{confusion_matrix(tgt_enc, pred_enc, labels=enc.transform(enc.classes_))}")

    # --- Task 3: Multi-label ---
    def parse_tech(x):
        if pd.isna(x) or str(x).strip().lower() in ["none", "nan", ""]:
            return ["none"]
        return [t.strip().lower() for t in str(x).split(",") if t.strip()]

    gt_tech   = result_df["Manipulation Techniques"].apply(parse_tech)
    pred_tech = result_df["Techniques_Pred"].apply(parse_tech)

    all_labels = sorted(set(
        l for lst in list(gt_tech) + list(pred_tech) for l in lst
        if l != "none"
    ))

    if all_labels:
        mlb    = MultiLabelBinarizer()
        mlb.fit([all_labels])
        y_true = mlb.transform(gt_tech)
        y_pred = mlb.transform(pred_tech)

        print("\n--- Task 3: Multi-label Classification ---")
        print(f"Classes: {mlb.classes_}")
        print(f"Exact Match (Accuracy): {accuracy_score(y_true, y_pred):.4f}")
        print(f"Precision (Macro):      {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
        print(f"Recall    (Macro):      {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
        print(f"F1 Score  (Macro):      {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
        print(f"Jaccard   (Samples):    {jaccard_score(y_true, y_pred, average='samples', zero_division=0):.4f}")


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# RUN 1: PRETRAINED (Zero-shot, chưa finetune)

In [11]:
print("\nLoading PRETRAINED tokenizer + model...")
tokenizer_pre = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer_pre.pad_token    = tokenizer_pre.eos_token
tokenizer_pre.padding_side = "right"

base_model_pre = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

pipe_pretrained = pipeline(
    "text-generation",
    model=base_model_pre,
    tokenizer=tokenizer_pre,
    device_map="auto",
)
print("Pretrained pipeline ready.")


Loading PRETRAINED tokenizer + model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


Pretrained pipeline ready.


In [12]:
results_pretrained = process_and_save(
    pipe_pretrained, test_df, tokenizer_pre,
    output_pretrained, "Pretrained LLM Zero-shot"
)
evaluate(results_pretrained, "Pretrained LLM Zero-shot")


  Running inference: Pretrained LLM Zero-shot
Starting analysis...


Pretrained LLM Zero-shot:   0%|          | 0/155 [00:00<?, ?it/s]

Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['framing the narrative', 'evasion', 'dismissal', 'emotional appeal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['framing the narrative', 'evasion', 'dismissal', 'emotional appeal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['None']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['emotional appeal', 'framing the narrative', 'character attack', 'playing the victim', 'guilt tripping', 'dismissal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['framing the narrative', 'evasion', '

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['persuasion', 'emotional appeal', 'framing the narrative', 'dismissal', 'minimization']
--------------------------------------------------------------------------------

Processed 10/155 dialogues
Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['framing the narrative', 'evasion', 'minimization', 'emotional appeal', 'deflection', 'dismissal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['framing the narrative', 'emotional appeal', 'dismissal', 'guilt tripping', 'minimization', 'evasion']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['emotional appeal', 'framing the narrative', 'character attack', 'minimization', 'guilt tripping', 'playing the victim', 'evasion', 'deflec

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:895: UserWarning: unknown class(es) ['none'] will be ignored
  warnings.warn(


In [13]:
del pipe_pretrained, base_model_pre
torch.cuda.empty_cache()
print("\nVRAM cleared.")


VRAM cleared.


# RUN 2: FINETUNED (Zero-shot, đã finetune)

In [14]:
print("\nLoading FINETUNED tokenizer + model...")
tokenizer_ft = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer_ft.pad_token    = tokenizer_ft.eos_token
tokenizer_ft.padding_side = "right"

base_model_ft = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
ft_model = PeftModel.from_pretrained(base_model_ft, adapter_path)

pipe_finetuned = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=tokenizer_ft,
    device_map="auto",
)
print("Finetuned pipeline ready.")


Loading FINETUNED tokenizer + model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Device set to use cuda:0


Finetuned pipeline ready.


In [15]:
results_finetuned = process_and_save(
    pipe_finetuned, test_df, tokenizer_ft,
    output_finetuned, "Finetuned LLM Zero-shot"
)
evaluate(results_finetuned, "Finetuned LLM Zero-shot")


  Running inference: Finetuned LLM Zero-shot
Starting analysis...


Finetuned LLM Zero-shot:   0%|          | 0/155 [00:00<?, ?it/s]

Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['evasion', 'emotional appeal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['playing the victim', 'emotional appeal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------

Processed 5/155 dialogues
Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['evasion', 'deflection']
-----------------------

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:895: UserWarning: unknown class(es) ['none'] will be ignored
  warnings.warn(
